In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio httpx kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp02
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import logging
import os
from dataclasses import dataclass, field, replace
from pathlib import Path
from typing import Any, Literal

import numpy as np
import polars as pl
from dotenv import load_dotenv

logger = logging.getLogger(__name__)


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Return the best available compute device as a ``torch.device``.

    Routes through ``kailash_ml.device()`` (the canonical kailash-ml 0.12+
    accelerator detector) so MPS / CUDA / ROCm / Intel XPU / CPU are all
    selected by the same logic the rest of the platform uses. The previous
    hand-rolled ``mps→cuda→cpu`` cascade is replaced because:

      * kailash-ml's detector knows about ROCm, Intel XPU, and fp16/bf16
        capability flags — the cascade in this helper did not.
      * Apple-Silicon students get the Metal Performance Shaders backend
        with mixed-precision (fp16) without any opt-in.
      * One detection point means lessons that print "Using device: …"
        agree with what kailash-ml's MLEngine() actually picks.
    """
    import kailash_ml as km
    import torch

    backend = km.device()  # BackendInfo (auto MPS on Mac, CUDA on Linux+NVIDIA, …)
    return torch.device(backend.device_string)


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ════════════════════════════════════════════════════════════════════════
# SPLIT FIRST, THEN PREPROCESS
# ════════════════════════════════════════════════════════════════════════
# kailash-ml's ``PreprocessingPipeline.setup()`` fits its imputer, its
# categorical encoder (including TARGET encoding) and its scaler on EVERY
# row it is given, and only afterwards splits train/test. Handing it the
# whole dataset therefore lets test-set statistics leak into training.
#
# The helpers below hold the test rows out FIRST, fit the pipeline on the
# training rows only, and push the held-out rows through the fitted
# ``transform()``. ``setup()`` still splits internally, but that split now
# happens inside the training rows, so it cannot leak anything.

_SETUP_OPTIONS: frozenset[str] = frozenset(
    {
        "normalize",
        "normalize_method",
        "categorical_encoding",
        "imputation_strategy",
        "impute_n_neighbors",
        "remove_outliers",
        "outlier_threshold",
        "max_cardinality",
        "exclude_columns",
        "remove_multicollinearity",
        "multicollinearity_threshold",
        "pca",
        "pca_components",
        "transform_target",
        "fix_imbalance",
        "imbalance_method",
    }
)

# Arguments the helpers own themselves; passing them through would
# re-introduce the leak (``data`` = the full frame) or fight the split.
_OWNED_BY_HELPER: frozenset[str] = frozenset({"data", "target", "train_size", "seed"})


@dataclass(frozen=True)
class PreprocessResult:
    """Outcome of :func:`split_then_preprocess` / :func:`preprocess_train_test`.

    Mirrors the fields of kailash-ml's ``SetupResult`` that exercises use,
    plus the fitted pipeline and a record of what was done to which split.

    ``train_data`` / ``test_data``
        Both produced by the SAME pipeline, fitted on the training rows only.
    ``transformers``
        The fitted imputer / encoder / scaler state (e.g.
        ``transformers["ordinal_mappings"]``), learned from training rows.
    ``train_only_steps``
        Steps applied to the training split only (outlier removal,
        class-imbalance resampling). The test split is never filtered or
        resampled — it stays a faithful sample of the data.
    ``test_null_fills``
        Columns that were complete in the training rows but have gaps in
        the test rows (the fitted pipeline has no learned fill for them),
        mapped to the TRAINING statistic used to fill them.
    """

    train_data: pl.DataFrame
    test_data: pl.DataFrame
    target_column: str
    task_type: str
    numeric_columns: list[str]
    categorical_columns: list[str]
    transformers: dict[str, Any]
    pipeline: Any
    original_shape: tuple[int, int]
    train_raw_rows: int
    test_raw_rows: int
    stratified: bool
    train_only_steps: tuple[str, ...] = ()
    test_null_fills: dict[str, Any] = field(default_factory=dict)
    summary: str = ""


def _looks_like_classification(series: pl.Series) -> bool:
    """Same rule kailash-ml uses to call a target "classification"."""
    if series.dtype in (pl.Boolean, pl.Categorical, pl.Utf8, pl.String):
        return True
    return series.drop_nulls().n_unique() <= 20


def split_raw_train_test(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
) -> tuple[pl.DataFrame, pl.DataFrame, bool]:
    """Hold out test rows from the RAW frame before anything is fitted.

    Rows are shuffled with ``seed``. ``test_size`` is a fraction in (0, 1)
    or an exact row count. With ``stratify="auto"`` a classification target
    (same rule as kailash-ml) is stratified so every class keeps its share
    in both splits; ``True`` / ``False`` force the choice.

    Returns ``(train_raw, test_raw, stratified)``; both splits keep the
    shuffled order.
    """
    if target not in df.columns:
        raise ValueError(f"Target column '{target}' not found in data.")
    n = df.height
    if n < 2:
        raise ValueError(f"Need at least 2 rows to split, got {n}.")
    if isinstance(test_size, np.integer):
        test_size = int(test_size)
    if isinstance(test_size, bool) or not isinstance(test_size, (int, float)):
        raise TypeError(
            f"test_size must be a fraction in (0, 1) or a row count, got {test_size!r}."
        )
    if isinstance(test_size, int):
        if not 1 <= test_size < n:
            raise ValueError(f"test_size as a row count must be in [1, {n - 1}], got {test_size}.")
        test_fraction = test_size / n
    else:
        if not 0.0 < test_size < 1.0:
            raise ValueError(f"test_size as a fraction must be in (0, 1), got {test_size}.")
        test_fraction = float(test_size)

    is_classification = _looks_like_classification(df[target])
    if stratify == "auto":
        stratified = is_classification
    elif stratify is True:
        if not is_classification:
            raise ValueError(
                f"stratify=True needs a classification target; '{target}' has "
                f"{df[target].drop_nulls().n_unique()} distinct values."
            )
        stratified = True
    elif stratify is False:
        stratified = False
    else:
        raise TypeError(f"stratify must be True, False or 'auto', got {stratify!r}.")

    perm = np.random.RandomState(seed).permutation(n)
    shuffled = df[perm.tolist()]

    if stratified:
        # Within each class (in shuffled order), the first round(n_c * f)
        # rows go to test — the class share is the same in both splits.
        is_test = pl.int_range(pl.len()).over(target) < (
            pl.len().over(target) * test_fraction
        ).round()
        flagged = shuffled.with_columns(is_test.alias("__split_is_test"))
        test_raw = flagged.filter(pl.col("__split_is_test")).drop("__split_is_test")
        train_raw = flagged.filter(~pl.col("__split_is_test")).drop("__split_is_test")
    else:
        n_test = test_size if isinstance(test_size, int) else int(round(n * test_fraction))
        train_raw = shuffled.head(n - n_test)
        test_raw = shuffled.tail(n_test)

    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError(
            f"Split produced an empty side (train={train_raw.height}, "
            f"test={test_raw.height}); adjust test_size."
        )
    return train_raw, test_raw, stratified


def _fill_unfitted_test_nulls(
    test_raw: pl.DataFrame,
    train_raw: pl.DataFrame,
    fitted: Any,
    strategy: str,
) -> tuple[pl.DataFrame, dict[str, Any]]:
    """Fill test gaps the fitted pipeline has no learned value for.

    kailash-ml only stores a fill value for columns that had gaps in the
    rows it was fitted on. A column that is complete in the training rows
    but has gaps in the test rows would otherwise reach the model as null.
    The fill is the TRAINING statistic for the same strategy (never a test
    statistic), and every such column is logged and returned.
    """
    if strategy == "drop":
        return test_raw, {}  # transform() drops incomplete rows itself
    learned = set(fitted.transformers.get("imputer_stats", {}))
    if strategy in ("knn", "iterative"):
        learned |= set(fitted.transformers.get("sklearn_imputer_cols", []))

    fills: dict[str, Any] = {}
    for col in fitted.numeric_columns + fitted.categorical_columns:
        if col in learned or col not in test_raw.columns:
            continue
        if test_raw[col].null_count() == 0:
            continue
        train_col = train_raw[col].drop_nulls()
        if col in fitted.categorical_columns or strategy == "mode":
            modes = train_col.mode().sort()
            if len(modes) == 0:
                raise ValueError(f"Cannot fill '{col}': no observed training values.")
            value = modes[0]
        elif strategy == "median":
            value = train_col.median()
        else:  # mean (and the numeric fallback for any other strategy)
            value = train_col.mean()
        if value is None:
            raise ValueError(f"Cannot fill '{col}': no observed training values.")
        fills[col] = value
    if fills:
        logger.warning(
            "Test rows have gaps in columns that were complete in training; "
            "filled with TRAINING statistics: %s",
            fills,
        )
        test_raw = test_raw.with_columns(
            [pl.col(c).fill_null(v) for c, v in fills.items()]
        )
    return test_raw, fills


def preprocess_train_test(
    train_raw: pl.DataFrame,
    test_raw: pl.DataFrame,
    target: str,
    *,
    seed: int = 42,
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Fit ``PreprocessingPipeline`` on ``train_raw`` only; transform both.

    ``setup_kwargs`` are ``PreprocessingPipeline.setup()`` options
    (``normalize``, ``categorical_encoding``, ``imputation_strategy`` …).
    Steps that change the ROWS of a split are applied to training rows only:

    * ``remove_outliers`` — outlier rows are dropped from training only.
    * ``fix_imbalance`` — SMOTE / ADASYN (or the class-weight flag) is
      applied to the whole transformed training split, never to test.

    Use this directly inside cross-validation folds; use
    :func:`split_then_preprocess` for a single train/test split.
    """
    from kailash_ml import PreprocessingPipeline

    owned = sorted(_OWNED_BY_HELPER & set(setup_kwargs))
    if owned:
        raise TypeError(
            f"{owned} are set by the split-first helper itself and cannot be "
            "passed through to setup()."
        )
    unknown = sorted(set(setup_kwargs) - _SETUP_OPTIONS)
    if unknown:
        raise TypeError(f"Unknown PreprocessingPipeline.setup() options: {unknown}.")
    for name, frame in (("train_raw", train_raw), ("test_raw", test_raw)):
        if target not in frame.columns:
            raise ValueError(f"Target column '{target}' not found in {name}.")
    if train_raw.height == 0 or test_raw.height == 0:
        raise ValueError("train_raw and test_raw must both have rows.")
    if set(train_raw.columns) != set(test_raw.columns):
        raise ValueError("train_raw and test_raw must have the same columns.")

    fix_imbalance = bool(setup_kwargs.pop("fix_imbalance", False))
    imbalance_method = setup_kwargs.get("imbalance_method", "smote")
    remove_outliers = bool(setup_kwargs.get("remove_outliers", False))

    pipeline = PreprocessingPipeline()
    # The pipeline only ever sees training rows. Its internal split stays
    # inside them; resampling is applied below to the full training split.
    fitted = pipeline.setup(
        data=train_raw, target=target, seed=seed, fix_imbalance=False, **setup_kwargs
    )

    train_only: list[str] = []
    if remove_outliers:
        # setup() removed the outlier rows (training rows only); its two
        # internal pieces together are every surviving training row.
        train_data = pl.concat([fitted.train_data, fitted.test_data], how="vertical")
        train_only.append(
            f"remove_outliers: {train_raw.height - train_data.height} training rows dropped"
        )
    else:
        train_data = pipeline.transform(train_raw)

    strategy = setup_kwargs.get("imputation_strategy", "mean")
    test_filled, test_fills = _fill_unfitted_test_nulls(test_raw, train_raw, fitted, strategy)
    test_data = pipeline.transform(test_filled)

    if fix_imbalance and fitted.task_type == "classification":
        resample = getattr(pipeline, "_apply_imbalance_correction", None)
        if resample is None:
            raise RuntimeError(
                "This kailash-ml version has no imbalance-correction step on "
                "PreprocessingPipeline; cannot apply fix_imbalance to the training split."
            )
        before = train_data.height
        train_data = resample(train_data, target, imbalance_method, seed)
        train_only.append(
            f"fix_imbalance ({imbalance_method}): training rows {before} -> {train_data.height}"
        )

    summary = "\n".join(
        [
            f"Task type: {fitted.task_type}",
            f"Split first: {train_raw.height} train rows / {test_raw.height} test rows "
            "(pipeline fitted on the train rows only)",
            f"Numeric features: {len(fitted.numeric_columns)}",
            f"Categorical features: {len(fitted.categorical_columns)}",
            f"Encoding: {setup_kwargs.get('categorical_encoding', 'onehot')}",
            f"Normalization: {'yes' if setup_kwargs.get('normalize', True) else 'no'}",
            f"Imputation: {strategy}",
            f"Train-only steps: {'; '.join(train_only) if train_only else 'none'}",
            f"Transformed train: {train_data.height} rows x {train_data.width} cols",
            f"Transformed test: {test_data.height} rows x {test_data.width} cols",
        ]
    )

    return PreprocessResult(
        train_data=train_data,
        test_data=test_data,
        target_column=target,
        task_type=fitted.task_type,
        numeric_columns=list(fitted.numeric_columns),
        categorical_columns=list(fitted.categorical_columns),
        transformers=dict(fitted.transformers),
        pipeline=pipeline,
        original_shape=(train_raw.height + test_raw.height, train_raw.width),
        train_raw_rows=train_raw.height,
        test_raw_rows=test_raw.height,
        stratified=False,
        train_only_steps=tuple(train_only),
        test_null_fills=test_fills,
        summary=summary,
    )


def split_then_preprocess(
    df: pl.DataFrame,
    target: str,
    *,
    test_size: float | int = 0.2,
    seed: int = 42,
    stratify: bool | Literal["auto"] = "auto",
    **setup_kwargs: Any,
) -> PreprocessResult:
    """Leak-free replacement for ``PreprocessingPipeline().setup(df, ...)``.

    1. Split the RAW frame first (seeded shuffle; stratified on the target
       for classification when ``stratify="auto"``).
    2. Fit ``PreprocessingPipeline`` on the training rows only.
    3. ``transform()`` the test rows with the fitted pipeline.

    ``setup_kwargs`` are passed to ``PreprocessingPipeline.setup()``; see
    :func:`preprocess_train_test` for how row-changing options are handled.
    """
    train_raw, test_raw, stratified = split_raw_train_test(
        df, target, test_size=test_size, seed=seed, stratify=stratify
    )
    result = preprocess_train_test(train_raw, test_raw, target, seed=seed, **setup_kwargs)
    split_line = "stratified on the target" if stratified else "random"
    return replace(
        result,
        stratified=stratified,
        summary=result.summary.replace("Split first:", f"Split first ({split_line}):", 1),
    )


# ── shared/data_loader.py ──
"""Unified data loading for MLFP course — supports local and Colab."""


import logging
from pathlib import Path

import polars as pl

logger = logging.getLogger(__name__)

# Google Drive shared folder containing all MLFP datasets
_DRIVE_FOLDER_ID = "16c3RkGmiwMWbjD7cJKbJx-JRZlgmQdws"

# Module subfolders on the shared Drive
_MODULES = {
    "mlfp01",
    "mlfp02",
    "mlfp03",
    "mlfp04",
    "mlfp05",
    "mlfp06",
    "mlfp_assessment",
}


def _is_colab() -> bool:
    """Detect if running inside Google Colab."""
    try:
        import google.colab  # noqa: F401

        return True
    except ImportError:
        return False


def _colab_data_root() -> Path:
    """Return the Drive-mounted mlfp_data path in Colab."""
    return Path("/content/drive/MyDrive/mlfp_data")


def _local_cache_dir() -> Path:
    """Return local cache directory for downloaded files."""
    cache = Path.cwd() / ".data_cache"
    cache.mkdir(exist_ok=True)
    return cache


def _download_from_drive(module: str, filename: str, dest: Path) -> Path:
    """Download a file from the shared Google Drive using gdown."""
    import gdown

    dest_dir = dest / module
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest_file = dest_dir / filename

    if dest_file.exists():
        logger.debug("Using cached file: %s", dest_file)
        return dest_file

    # gdown can download from a folder by file path
    url = f"https://drive.google.com/drive/folders/{_DRIVE_FOLDER_ID}"
    logger.info("Downloading %s/%s from Google Drive...", module, filename)

    # Download the specific file from the shared folder
    try:
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
            remaining_ok=True,
        )
    except TypeError:
        # Older gdown versions don't support remaining_ok
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
        )

    if not dest_file.exists():
        # Try direct download if folder download didn't isolate the file
        for candidate in dest.rglob(filename):
            if candidate.is_file():
                if candidate != dest_file:
                    candidate.rename(dest_file)
                return dest_file

        msg = (
            f"File not found after download: {module}/{filename}. "
            f"Check that it exists in the mlfp_data shared Drive."
        )
        raise FileNotFoundError(msg)

    return dest_file


def _read_file(path: Path) -> pl.DataFrame:
    """Read a data file into a polars DataFrame."""
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pl.read_csv(path, try_parse_dates=True)
    elif suffix == ".parquet":
        return pl.read_parquet(path)
    elif suffix == ".json":
        return pl.read_json(path)
    elif suffix in (".p", ".pickle", ".pkl"):
        import pickle

        with open(path, "rb") as f:
            obj = pickle.load(f)  # noqa: S301
        if isinstance(obj, pl.DataFrame):
            return obj
        raise TypeError(
            f"Cannot convert pickle object of type {type(obj)} to polars DataFrame. "
            f"Convert the pickle to parquet upstream: pl.from_pandas(obj).write_parquet('out.parquet')"
        )
    else:
        raise ValueError(
            f"Unsupported file format: {suffix}. Use .csv, .parquet, or .json"
        )


def _repo_data_dir() -> Path | None:
    """Find the repo-local data/ directory by walking up from cwd."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data"
        if candidate.is_dir() and (parent / "pyproject.toml").exists():
            return candidate
    return None


class MLFPDataLoader:
    """Load MLFP course datasets with automatic source resolution.

    Resolution order:
    1. Colab: Drive mount at /content/drive/MyDrive/mlfp_data/
    2. Local repo data/ directory (committed datasets)
    3. Google Drive download via gdown (cached in .data_cache/)

    Usage:
        loader = MLFPDataLoader()
        df = loader.load("mlfp01", "hdbprices.csv")

    Shortcut:
        df = MLFPDataLoader.mlfp01("hdbprices.csv")
    """

    def __init__(self, cache_dir: Path | str | None = None):
        self._colab = _is_colab()
        if self._colab:
            self._root = _colab_data_root()
        else:
            self._local_data = _repo_data_dir()
            self._cache = Path(cache_dir) if cache_dir else _local_cache_dir()

    def load_raw(self, module: str, filename: str) -> Path:
        """Return the file path without reading into memory.

        Use this for image directories, audio files, or any data that torch/HF
        loads directly rather than via polars.

        Args:
            module: Module subfolder (e.g., "mlfp05")
            filename: File or directory name (e.g., "fashion_mnist", "cifar10")

        Returns:
            Path to the local file or directory.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
        else:
            if self._local_data:
                local_path = self._local_data / module / filename
                if local_path.exists():
                    return local_path
            path = self._cache / module / filename

        if not path.exists():
            raise FileNotFoundError(
                f"Raw data not found: {module}/{filename}. "
                f"Run 'python scripts/fetch-real-data.py' to download."
            )
        return path

    @staticmethod
    def load_hf(
        dataset_name: str,
        split: str = "train",
        streaming: bool = False,
    ):
        """Load a HuggingFace dataset directly (not via polars).

        Use this for large datasets (millions of rows) or multimodal data
        (images, audio) that don't fit into a DataFrame.

        Args:
            dataset_name: HuggingFace dataset ID (e.g., "zalando-datasets/fashion_mnist")
            split: Dataset split ("train", "test", "validation")
            streaming: If True, returns an IterableDataset for memory-efficient processing

        Returns:
            HuggingFace Dataset or IterableDataset object.
        """
        from datasets import load_dataset

        logger.info(
            "Loading HuggingFace dataset: %s (split=%s, streaming=%s)",
            dataset_name,
            split,
            streaming,
        )
        return load_dataset(dataset_name, split=split, streaming=streaming)

    def load(self, module: str, filename: str) -> pl.DataFrame:
        """Load a dataset file as a polars DataFrame.

        Args:
            module: Module subfolder (e.g., "mlfp01", "mlfp_assessment")
            filename: File name within the module folder (e.g., "hdbprices.csv")

        Returns:
            polars DataFrame with the loaded data.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
            if not path.exists():
                raise FileNotFoundError(
                    f"File not found: {path}. "
                    f"Ensure mlfp_data is accessible in your Google Drive."
                )
        else:
            # Check repo-local data/ first, then fall back to Drive download
            if self._local_data:
                module_dir = self._local_data / module
                local_path = module_dir / filename
                if local_path.exists():
                    path = local_path
                    logger.info(
                        "Loading %s/%s from local data/ (%s)", module, filename, path
                    )
                    return _read_file(path)
                # Bundled data is present for this module but the file is absent
                # → it is genuinely missing. Raise immediately instead of falling
                # back to a Drive download that would hang offline (and pulls the
                # whole folder online just to discover the file is not there).
                if module_dir.is_dir():
                    raise FileNotFoundError(
                        f"File not found: {module}/{filename} "
                        f"(not in local data/{module}/)."
                    )
            path = _download_from_drive(module, filename, self._cache)

        logger.info("Loading %s/%s (%s)", module, filename, path)
        return _read_file(path)

    def list_files(self, module: str) -> list[str]:
        """List available data files in a module folder."""
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            root = self._root / module
        else:
            root = self._cache / module

        if not root.exists():
            return []

        return sorted(f.name for f in root.iterdir() if f.is_file())

    # -- Module shortcuts --

    @classmethod
    def mlfp01(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp01 (Data Pipelines & Visualisation)."""
        return cls().load("mlfp01", filename)

    @classmethod
    def mlfp02(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp02 (Statistical Mastery)."""
        return cls().load("mlfp02", filename)

    @classmethod
    def mlfp03(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp03 (Supervised ML)."""
        return cls().load("mlfp03", filename)

    @classmethod
    def mlfp04(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp04 (Unsupervised ML)."""
        return cls().load("mlfp04", filename)

    @classmethod
    def mlfp05(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp05 (Deep Learning & Vision)."""
        return cls().load("mlfp05", filename)

    @classmethod
    def mlfp06(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp06 (LLMs, Agents & Transformation)."""
        return cls().load("mlfp06", filename)

    @classmethod
    def assessment(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp_assessment (capstone datasets)."""
        return cls().load("mlfp_assessment", filename)


# ── shared/mlfp02/ex_7.py ──
"""
Shared infrastructure for MLFP02 Exercise 7 — CUPED and Causal Inference.

Contains: experiment data loading, designed-allocation SRM checks, naive
A/B baseline, CUPED reference helpers, Bayesian decision utilities, mSPRT
helpers, DiD panel simulator and parallel-trends test. Technique-specific narration and
checkpoints live in the per-technique files.

Importable from any cwd after `uv sync`:

"""

from pathlib import Path
from typing import Any

import numpy as np
import polars as pl
from scipy import stats


# ════════════════════════════════════════════════════════════════════════
# ENVIRONMENT SETUP
# ════════════════════════════════════════════════════════════════════════

setup_environment()

# Output directory for visualisation artifacts
OUTPUT_DIR = Path("outputs") / "ex7_causal"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ════════════════════════════════════════════════════════════════════════
# DATA LOADING — Experiment data with pre-experiment covariates
# ════════════════════════════════════════════════════════════════════════


# The experiment was DESIGNED as a four-arm test with this traffic split.
# (variant_c was meant to receive 10% of users; a bucketing bug sent it 15%.)
# SRM checks must compare observed counts against THIS design, not 50/50.
DESIGNED_ALLOCATION: dict[str, float] = {
    "control": 0.40,
    "treatment_a": 0.35,
    "treatment_b": 0.15,
    "variant_c": 0.10,
}

# Exercise 7 analyses ONE treatment arm against control. Pooling arms would
# estimate a mixture of three different treatments, which answers no
# product question.
ANALYSIS_ARM = "treatment_a"


def load_experiment() -> pl.DataFrame:
    """Load the MLFP02 experiment dataset.

    Columns: user_id, experiment_group, metric_value, pre_metric_value,
    revenue, timestamp, segment, platform, country.

    ``pre_metric_value`` is measured BEFORE assignment (a valid CUPED
    covariate). ``metric_value`` is measured DURING the experiment and is
    affected by treatment — it must never be used as a CUPED covariate.
    """
    loader = MLFPDataLoader()
    return loader.load("mlfp02", "experiment_data.parquet")


def split_groups(
    experiment: pl.DataFrame, treatment_arm: str = ANALYSIS_ARM
) -> tuple[pl.DataFrame, pl.DataFrame]:
    """Return (control, treatment) sub-frames for control vs ONE named arm."""
    arms = set(experiment["experiment_group"].unique().to_list())
    if treatment_arm not in arms:
        raise ValueError(f"Unknown arm {treatment_arm!r}; available: {sorted(arms)}")
    control = experiment.filter(pl.col("experiment_group") == "control")
    treatment = experiment.filter(pl.col("experiment_group") == treatment_arm)
    return control, treatment


def get_revenue_arrays(
    control: pl.DataFrame, treatment: pl.DataFrame
) -> tuple[np.ndarray, np.ndarray]:
    """Extract revenue arrays as float64 numpy arrays."""
    y_c = control["revenue"].to_numpy().astype(np.float64)
    y_t = treatment["revenue"].to_numpy().astype(np.float64)
    return y_c, y_t


def get_covariate_arrays(
    control: pl.DataFrame, treatment: pl.DataFrame, column: str = "pre_metric_value"
) -> tuple[np.ndarray, np.ndarray]:
    """Extract a pre-experiment covariate as float64 numpy arrays."""
    x_c = control[column].to_numpy().astype(np.float64)
    x_t = treatment[column].to_numpy().astype(np.float64)
    return x_c, x_t


# ════════════════════════════════════════════════════════════════════════
# SAMPLE RATIO MISMATCH (SRM)
# ════════════════════════════════════════════════════════════════════════


def srm_allocation_check(
    experiment: pl.DataFrame,
    allocation: dict[str, float] | None = None,
) -> dict[str, Any]:
    """Chi-square SRM test of ALL arms against the designed allocation.

    Returns the chi-square statistic, p-value, and a per-arm table of
    observed vs expected counts with standardised residuals
    (observed - expected) / sqrt(expected), which localise the faulty arm.
    """
    allocation = allocation or DESIGNED_ALLOCATION
    counts = experiment.group_by("experiment_group").len()
    observed = {r["experiment_group"]: int(r["len"]) for r in counts.iter_rows(named=True)}
    arms = list(allocation)
    total = sum(observed.get(a, 0) for a in arms)
    obs = np.array([observed.get(a, 0) for a in arms], dtype=np.float64)
    exp = np.array([allocation[a] for a in arms]) * total
    chi2, p = stats.chisquare(obs, f_exp=exp)
    residuals = (obs - exp) / np.sqrt(exp)
    table = [
        {
            "arm": a,
            "observed": int(o),
            "expected": float(e),
            "observed_share": float(o / total),
            "designed_share": float(allocation[a]),
            "std_residual": float(r),
        }
        for a, o, e, r in zip(arms, obs, exp, residuals)
    ]
    return {"chi2": float(chi2), "p_value": float(p), "arms": table}


def compute_srm(
    n_c: int,
    n_t: int,
    design_c: float = DESIGNED_ALLOCATION["control"],
    design_t: float = DESIGNED_ALLOCATION[ANALYSIS_ARM],
) -> float:
    """Pairwise chi-square SRM test against the DESIGNED ratio design_c:design_t.

    Returns the p-value; p < 0.01 indicates sample ratio mismatch and the
    comparison must not be analysed until the cause is found.
    """
    observed = np.array([n_c, n_t], dtype=np.float64)
    share_c = design_c / (design_c + design_t)
    expected = (n_c + n_t) * np.array([share_c, 1 - share_c])
    _, srm_p = stats.chisquare(observed, f_exp=expected)
    return float(srm_p)


# ════════════════════════════════════════════════════════════════════════
# STANDARD A/B (BASELINE)
# ════════════════════════════════════════════════════════════════════════


def naive_ab(y_c: np.ndarray, y_t: np.ndarray) -> dict[str, float]:
    """Standard Welch-style lift, SE, 95% CI, z, two-sided p-value."""
    n_c, n_t = len(y_c), len(y_t)
    mean_c, mean_t = y_c.mean(), y_t.mean()
    lift = mean_t - mean_c
    se = float(np.sqrt(y_c.var(ddof=1) / n_c + y_t.var(ddof=1) / n_t))
    ci_lo = lift - 1.96 * se
    ci_hi = lift + 1.96 * se
    z = lift / se if se > 0 else 0.0
    p = 2 * (1 - stats.norm.cdf(abs(z)))
    return {
        "mean_c": float(mean_c),
        "mean_t": float(mean_t),
        "lift": float(lift),
        "se": se,
        "ci_lo": float(ci_lo),
        "ci_hi": float(ci_hi),
        "z": float(z),
        "p_value": float(p),
    }


# ════════════════════════════════════════════════════════════════════════
# CUPED — SINGLE COVARIATE
# ════════════════════════════════════════════════════════════════════════


def single_cov_cuped(
    y_c: np.ndarray, y_t: np.ndarray, x_c: np.ndarray, x_t: np.ndarray
) -> dict[str, Any]:
    """Single-covariate CUPED: Y_adj = Y - theta*(X - E[X]).

    theta = Cov(Y, X) / Var(X). Returns adjusted arrays, point estimate,
    SE, CI, p-value, theta, rho, and variance reduction.
    """
    x_all = np.concatenate([x_c, x_t])
    y_all = np.concatenate([y_c, y_t])
    var_x = np.var(x_all, ddof=1)
    theta = np.cov(y_all, x_all)[0, 1] / var_x if var_x > 0 else 0.0
    rho = np.corrcoef(y_all, x_all)[0, 1]
    x_mean = x_all.mean()

    y_c_adj = y_c - theta * (x_c - x_mean)
    y_t_adj = y_t - theta * (x_t - x_mean)

    n_c, n_t = len(y_c), len(y_t)
    lift_adj = y_t_adj.mean() - y_c_adj.mean()
    se = float(np.sqrt(y_c_adj.var(ddof=1) / n_c + y_t_adj.var(ddof=1) / n_t))
    ci_lo, ci_hi = lift_adj - 1.96 * se, lift_adj + 1.96 * se
    z = lift_adj / se if se > 0 else 0.0
    p = 2 * (1 - stats.norm.cdf(abs(z)))

    return {
        "theta": float(theta),
        "rho": float(rho),
        "y_c_adj": y_c_adj,
        "y_t_adj": y_t_adj,
        "lift": float(lift_adj),
        "se": se,
        "ci_lo": float(ci_lo),
        "ci_hi": float(ci_hi),
        "z": float(z),
        "p_value": float(p),
        "theoretical_reduction": float(rho**2),
    }


# ════════════════════════════════════════════════════════════════════════
# CUPED — MULTI-COVARIATE
# ════════════════════════════════════════════════════════════════════════


def multi_cov_cuped(
    y_c: np.ndarray,
    y_t: np.ndarray,
    X_c: np.ndarray,
    X_t: np.ndarray,
) -> dict[str, Any]:
    """Multi-covariate CUPED via OLS on centered covariates.

    theta = (X'X)^-1 X'Y — multivariate regression coefficients.
    """
    y_all = np.concatenate([y_c, y_t])
    X_all = np.vstack([X_c, X_t])
    X_mean = X_all.mean(axis=0)
    X_centered = X_all - X_mean
    theta = np.linalg.lstsq(X_centered, y_all - y_all.mean(), rcond=None)[0]

    y_c_adj = y_c - (X_c - X_mean) @ theta
    y_t_adj = y_t - (X_t - X_mean) @ theta

    n_c, n_t = len(y_c), len(y_t)
    lift = y_t_adj.mean() - y_c_adj.mean()
    se = float(np.sqrt(y_c_adj.var(ddof=1) / n_c + y_t_adj.var(ddof=1) / n_t))
    return {
        "theta": theta,
        "y_c_adj": y_c_adj,
        "y_t_adj": y_t_adj,
        "lift": float(lift),
        "se": se,
        "ci_lo": float(lift - 1.96 * se),
        "ci_hi": float(lift + 1.96 * se),
    }


# ════════════════════════════════════════════════════════════════════════
# CUPED — STRATIFIED
# ════════════════════════════════════════════════════════════════════════


def stratify_by_covariate(
    x_c: np.ndarray, x_t: np.ndarray, percentiles: tuple[int, int] = (33, 67)
) -> dict[str, np.ndarray]:
    """Build Low/Medium/High strata masks over concatenated (x_c, x_t)."""
    x_all = np.concatenate([x_c, x_t])
    q_lo, q_hi = np.percentile(x_all, percentiles)
    return {
        "Low spenders": (x_all <= q_lo),
        "Medium spenders": (x_all > q_lo) & (x_all <= q_hi),
        "High spenders": (x_all > q_hi),
    }


def stratified_cuped(
    y_c: np.ndarray,
    y_t: np.ndarray,
    x_c: np.ndarray,
    x_t: np.ndarray,
    strata: dict[str, np.ndarray],
    min_per_cell: int = 30,
) -> dict[str, dict[str, float]]:
    """Apply CUPED within each stratum. Returns {name: {n_ctrl,n_treat,lift,se,p}}."""
    n_c = len(y_c)
    results: dict[str, dict[str, float]] = {}
    for name, mask in strata.items():
        ctrl_mask, treat_mask = mask[:n_c], mask[n_c:]
        y_c_s, y_t_s = y_c[ctrl_mask], y_t[treat_mask]
        x_c_s, x_t_s = x_c[ctrl_mask], x_t[treat_mask]
        if len(y_c_s) < min_per_cell or len(y_t_s) < min_per_cell:
            continue
        x_all_s = np.concatenate([x_c_s, x_t_s])
        y_all_s = np.concatenate([y_c_s, y_t_s])
        var_x = np.var(x_all_s, ddof=1)
        theta_s = np.cov(y_all_s, x_all_s)[0, 1] / var_x if var_x > 0 else 0.0
        x_mean = x_all_s.mean()
        y_c_adj = y_c_s - theta_s * (x_c_s - x_mean)
        y_t_adj = y_t_s - theta_s * (x_t_s - x_mean)
        lift = y_t_adj.mean() - y_c_adj.mean()
        se = float(
            np.sqrt(y_c_adj.var(ddof=1) / len(y_c_s) + y_t_adj.var(ddof=1) / len(y_t_s))
        )
        z = lift / se if se > 0 else 0.0
        p = 2 * (1 - stats.norm.cdf(abs(z)))
        results[name] = {
            "n_ctrl": len(y_c_s),
            "n_treat": len(y_t_s),
            "lift": float(lift),
            "se": se,
            "p_value": float(p),
        }
    return results


# ════════════════════════════════════════════════════════════════════════
# BAYESIAN A/B
# ════════════════════════════════════════════════════════════════════════


def bayesian_decision(
    y_c_adj: np.ndarray,
    y_t_adj: np.ndarray,
    lift: float,
    practical_threshold: float = 1.0,
) -> dict[str, float]:
    """Bayesian posterior using normal approximation on CUPED-adjusted arrays.

    Returns P(treatment > control), P(treatment > control + threshold),
    expected loss (both directions), and credible interval.
    """
    n_c, n_t = len(y_c_adj), len(y_t_adj)
    se_c = y_c_adj.std(ddof=1) / np.sqrt(n_c)
    se_t = y_t_adj.std(ddof=1) / np.sqrt(n_t)
    se_lift = float(np.sqrt(se_c**2 + se_t**2))

    prob_better = float(1 - stats.norm.cdf(0, loc=lift, scale=se_lift))
    prob_practical = float(
        1 - stats.norm.cdf(practical_threshold, loc=lift, scale=se_lift)
    )

    # Posterior lift L ~ Normal(m = lift, s = se_lift). With z = m / s:
    #   loss if we ship treatment  = E[max(0, -L)] = s*phi(z) - m*Phi(-z)
    #   loss if we keep control    = E[max(0,  L)] = s*phi(z) + m*Phi(z)
    z = lift / se_lift
    exp_loss_treat = float(se_lift * stats.norm.pdf(z) - lift * stats.norm.cdf(-z))
    exp_loss_ctrl = float(se_lift * stats.norm.pdf(z) + lift * stats.norm.cdf(z))

    return {
        "prob_treatment_better": prob_better,
        "prob_practical": prob_practical,
        "expected_loss_treatment": exp_loss_treat,
        "expected_loss_control": exp_loss_ctrl,
        "se_lift": se_lift,
        "ci_lo": float(lift - 1.96 * se_lift),
        "ci_hi": float(lift + 1.96 * se_lift),
    }


def bayesian_decision_rule(prob_better: float, exp_loss_treat: float) -> str:
    """Simple ship/continue/hold decision rule."""
    if prob_better > 0.95 and exp_loss_treat < 0.50:
        return "SHIP — high confidence + low expected loss"
    if prob_better > 0.80:
        return "CONTINUE — promising but need more data"
    return "HOLD — insufficient evidence"


# ════════════════════════════════════════════════════════════════════════
# SEQUENTIAL TESTING — mSPRT
# ════════════════════════════════════════════════════════════════════════


def msprt_lambda(diff: float, v_n: float, tau_sq: float) -> float:
    """mSPRT mixture likelihood ratio for a Normal mean difference.

    Lambda_n = sqrt(V / (V + tau^2)) * exp(tau^2 * diff^2 / (2 V (V + tau^2)))
    where V is the variance of the difference estimate at the current look.
    """
    return float(
        np.sqrt(v_n / (v_n + tau_sq))
        * np.exp(tau_sq * diff**2 / (2 * v_n * (v_n + tau_sq)))
    )


def msprt_sequential_pvalues(
    experiment: pl.DataFrame,
    tau_sq: float,
    treatment_arm: str = ANALYSIS_ARM,
    min_per_group: int = 100,
    skip_first_days: int = 3,
) -> list[dict[str, float]]:
    """Walk the experiment day by day, computing fixed and mSPRT p-values.

    tau_sq is the mSPRT hyperparameter — typically set to se_naive**2.
    The always-valid p-value is the running minimum of 1/Lambda_n, so it can
    only go down as evidence accumulates.
    """
    if experiment["timestamp"].dtype in [pl.Utf8, pl.String]:
        exp_daily = experiment.with_columns(
            pl.col("timestamp")
            .str.to_datetime("%Y-%m-%d %H:%M:%S")
            .dt.date()
            .alias("day")
        )
    else:
        exp_daily = experiment.with_columns(
            pl.col("timestamp").cast(pl.Date).alias("day")
        )

    days = sorted(exp_daily["day"].unique().to_list())
    results: list[dict[str, float]] = []
    p_seq = 1.0
    for i, day in enumerate(days):
        if i < skip_first_days:
            continue
        cumulative = exp_daily.filter(pl.col("day") <= day)
        c = (
            cumulative.filter(pl.col("experiment_group") == "control")["revenue"]
            .to_numpy()
            .astype(np.float64)
        )
        t = (
            cumulative.filter(pl.col("experiment_group") == treatment_arm)["revenue"]
            .to_numpy()
            .astype(np.float64)
        )
        if len(c) < min_per_group or len(t) < min_per_group:
            continue
        diff = t.mean() - c.mean()
        v_n = c.var(ddof=1) / len(c) + t.var(ddof=1) / len(t)
        se = float(np.sqrt(v_n))
        z = diff / se if se > 0 else 0.0
        p_fixed = float(2 * (1 - stats.norm.cdf(abs(z))))
        # mSPRT always-valid p-value: running minimum of 1 / Lambda_n
        lambda_n = msprt_lambda(diff, v_n, tau_sq)
        p_seq = float(min(p_seq, 1.0 / lambda_n))
        results.append(
            {
                "day": i + 1,
                "n": int(len(c) + len(t)),
                "lift": float(diff),
                "p_fixed": p_fixed,
                "p_sequential": p_seq,
            }
        )
    return results


# ════════════════════════════════════════════════════════════════════════
# PEEKING PROBLEM SIMULATION
# ════════════════════════════════════════════════════════════════════════


def simulate_peeking(
    n_sims: int = 1000,
    n_per_sim: int = 2000,
    n_checks: int = 20,
    seed: int = 42,
) -> dict[str, float]:
    """Simulate A/A experiments (zero effect) with and without peeking.

    Returns dict with false-positive rates for: no-peek and fixed-p peeking.
    The looks are on ACCUMULATING data, so they are strongly correlated —
    the inflation must be measured by simulation; the independent-tests
    formula 1 - 0.95**k does not apply.
    """
    rng = np.random.default_rng(seed=seed)
    false_pos_fixed = 0
    false_pos_no_peek = 0

    for _ in range(n_sims):
        sim_ctrl = rng.normal(50, 10, size=n_per_sim)
        sim_treat = rng.normal(50, 10, size=n_per_sim)  # no real effect

        # no peeking
        z_end = (sim_treat.mean() - sim_ctrl.mean()) / np.sqrt(
            sim_ctrl.var(ddof=1) / n_per_sim + sim_treat.var(ddof=1) / n_per_sim
        )
        if 2 * (1 - stats.norm.cdf(abs(z_end))) < 0.05:
            false_pos_no_peek += 1

        # peeking at n_checks points, fixed p-values
        peeked_sig = False
        for check_n in np.linspace(100, n_per_sim, n_checks, dtype=int):
            sc = sim_ctrl[:check_n]
            st = sim_treat[:check_n]
            se_p = np.sqrt(sc.var(ddof=1) / check_n + st.var(ddof=1) / check_n)
            z_p = (st.mean() - sc.mean()) / se_p if se_p > 0 else 0
            if 2 * (1 - stats.norm.cdf(abs(z_p))) < 0.05:
                peeked_sig = True
                break
        if peeked_sig:
            false_pos_fixed += 1

    return {
        "n_sims": n_sims,
        "n_checks": n_checks,
        "rate_no_peek": false_pos_no_peek / n_sims,
        "rate_fixed_peek": false_pos_fixed / n_sims,
    }


# ════════════════════════════════════════════════════════════════════════
# DIFFERENCE-IN-DIFFERENCES — hypothetical HDB cooling measure (simulated)
# ════════════════════════════════════════════════════════════════════════


def simulate_hdb_cooling_panel(
    n_per_period: int = 200,
    n_pre: int = 6,
    n_post: int = 6,
    central_base: float = 550_000,
    noncentral_base: float = 450_000,
    growth_per_period: float = 2_000,
    central_extra_growth: float = 0.0,
    policy_effect: float = -20_000,
    seed: int = 99,
) -> pl.DataFrame:
    """Simulate quarterly HDB-style transactions around a HYPOTHETICAL measure.

    The scenario is illustrative: a cooling measure that applies only to
    Central-region flats (treated) and not to Non-Central flats (control).
    Both regions share a common price trend of ``growth_per_period``;
    ``central_extra_growth`` adds a Central-only pre-existing trend, which
    VIOLATES parallel trends (used to show the test can detect it).
    ``policy_effect`` is the true causal effect on Central prices after
    the measure (periods >= n_pre).

    Returns one row per transaction: period, central (0/1), post (0/1), price.
    """
    rng = np.random.default_rng(seed=seed)
    frames = []
    for t in range(n_pre + n_post):
        post = int(t >= n_pre)
        for central, base, sd in ((1, central_base, 80_000), (0, noncentral_base, 70_000)):
            mean = base + t * growth_per_period
            if central:
                mean += t * central_extra_growth + post * policy_effect
            prices = rng.normal(mean, sd, size=n_per_period)
            frames.append(
                pl.DataFrame(
                    {
                        "period": np.full(n_per_period, t, dtype=np.int64),
                        "central": np.full(n_per_period, central, dtype=np.int64),
                        "post": np.full(n_per_period, post, dtype=np.int64),
                        "price": prices,
                    }
                )
            )
    return pl.concat(frames)


def did_cells(panel: pl.DataFrame) -> dict[str, np.ndarray]:
    """Split the panel into the four DiD cells (group x pre/post) as arrays."""

    def cell(central: int, post: int) -> np.ndarray:
        return (
            panel.filter((pl.col("central") == central) & (pl.col("post") == post))[
                "price"
            ]
            .to_numpy()
            .astype(np.float64)
        )

    return {
        "pre_central": cell(1, 0),
        "post_central": cell(1, 1),
        "pre_noncentral": cell(0, 0),
        "post_noncentral": cell(0, 1),
    }


def diff_in_diff(cells: dict[str, np.ndarray]) -> dict[str, float]:
    """Compute DiD estimate, SE, CI, z, p from four cell arrays."""
    y_tp = cells["pre_central"].mean()
    y_tq = cells["post_central"].mean()
    y_cp = cells["pre_noncentral"].mean()
    y_cq = cells["post_noncentral"].mean()

    did = (y_tq - y_tp) - (y_cq - y_cp)
    se = float(np.sqrt(sum(arr.var(ddof=1) / len(arr) for arr in cells.values())))
    z = did / se if se > 0 else 0.0
    p = float(2 * (1 - stats.norm.cdf(abs(z))))
    return {
        "y_treat_pre": float(y_tp),
        "y_treat_post": float(y_tq),
        "y_ctrl_pre": float(y_cp),
        "y_ctrl_post": float(y_cq),
        "did_estimate": float(did),
        "se": se,
        "ci_lo": float(did - 1.96 * se),
        "ci_hi": float(did + 1.96 * se),
        "z": float(z),
        "p_value": p,
    }


# ════════════════════════════════════════════════════════════════════════
# PARALLEL TRENDS TEST
# ════════════════════════════════════════════════════════════════════════


def parallel_trends_test(panel: pl.DataFrame, alpha: float = 0.05) -> dict[str, Any]:
    """Test parallel PRE-period trends with a group x time interaction.

    Fits OLS on the pre-period transactions of the SAME panel used for DiD:
        price = b0 + b1*period + b2*central + b3*(central*period) + e
    b3 is the difference in pre-period slopes (Central minus Non-Central).
    H0: b3 = 0 (parallel trends). The null distribution is centred at zero,
    so a real slope difference produces a small p-value.
    """
    pre = panel.filter(pl.col("post") == 0)
    t = pre["period"].to_numpy().astype(np.float64)
    g = pre["central"].to_numpy().astype(np.float64)
    y = pre["price"].to_numpy().astype(np.float64)
    X = np.column_stack([np.ones_like(t), t, g, g * t])
    beta, *_ = np.linalg.lstsq(X, y, rcond=None)
    resid = y - X @ beta
    n, k = X.shape
    sigma2 = resid @ resid / (n - k)
    cov = sigma2 * np.linalg.inv(X.T @ X)
    se_b3 = float(np.sqrt(cov[3, 3]))
    t_stat = float(beta[3] / se_b3)
    p = float(2 * stats.t.sf(abs(t_stat), df=n - k))

    means = (
        pre.group_by(["period", "central"])
        .agg(pl.col("price").mean())
        .sort(["central", "period"])
    )
    pre_central = means.filter(pl.col("central") == 1)["price"].to_list()
    pre_noncentral = means.filter(pl.col("central") == 0)["price"].to_list()

    return {
        "pre_central": pre_central,
        "pre_noncentral": pre_noncentral,
        "time_points": np.arange(len(pre_central)),
        "slope_central": float(beta[1] + beta[3]),
        "slope_noncentral": float(beta[1]),
        "slope_diff": float(beta[3]),
        "slope_diff_se": se_b3,
        "t_stat": t_stat,
        "p_value": p,
        "passes": bool(p > alpha),
    }


# ════════════════════════════════════════════════════════════════════════
# VARIANCE REDUCTION REPORTING
# ════════════════════════════════════════════════════════════════════════


def variance_reduction(se_baseline: float, se_adjusted: float) -> dict[str, float]:
    """Report variance and CI-width reduction from baseline -> adjusted SE."""
    var_red = 1 - (se_adjusted**2) / (se_baseline**2)
    ci_red = 1 - (se_adjusted / se_baseline)
    return {
        "variance_reduction": float(var_red),
        "ci_width_reduction": float(ci_red),
        "effective_sample_multiplier": float(1 / max(1 - var_red, 1e-9)),
    }


def print_banner(title: str) -> None:
    """Consistent section header across technique files."""
    print("\n" + "=" * 70)
    print(f"  {title}")
    print("=" * 70)




# ════════════════════════════════════════════════════════════════════════
# MLFP02 — Exercise 7.3: Sequential Testing with mSPRT
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Implement sequential testing with mSPRT (always-valid p-values)
#   - Demonstrate the peeking problem with simulation
#   - Compare fixed vs sequential p-values over time
#   - Understand why standard p-values fail under continuous monitoring
#   - Log sequential results to ExperimentTracker
#
# PREREQUISITES: Exercise 7.1-7.2 (CUPED, Bayesian concepts)
# ESTIMATED TIME: ~45 min
#
# TASKS:
#   1. Load experiment data and compute baseline SE
#   2. Sequential testing: mSPRT always-valid p-values day by day
#   3. Peeking problem simulation: inflated Type I error
#   4. Compare fixed vs sequential p-value trajectories
#   5. Visualise both analyses
#   6. Apply to Singapore ride-hailing scenario
#
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations


import numpy as np
import plotly.graph_objects as go
from kailash_ml import ExperimentTracker



## THEORY — The Peeking Problem and Sequential Testing


In [ ]:
# Standard p-values are designed for a SINGLE look at the data. If you
# peek at your experiment 20 times and stop the first time p < 0.05, the
# actual false positive rate is far above 5% — Task 3 measures it.
#
# Why? Every peek is another chance for random noise to cross the line.
# The peeks are NOT independent tests: each look re-uses all the data of
# the previous look plus a little more, so consecutive z-statistics are
# highly correlated. That is why the naive formula 1 - 0.95^20 = 64% for
# 20 independent tests is wrong here; for 20 equally spaced looks the
# true rate is roughly 25% — still five times the promised 5%. Because
# there is no simple closed form, we measure it by simulation.
#
# mSPRT (mixture Sequential Probability Ratio Test) provides "always-
# valid" p-values that remain correct no matter when you look. The
# trade-off: mSPRT p-values are more conservative — they need more
# data to reach significance. But they never lie.
#
# Analogy: Fixed p-values are like a bathroom scale that only gives the
# right weight if you step on it exactly once. Step on it 20 times and
# take the lowest reading? You'll think you lost weight. mSPRT is a
# scale that gives the correct reading no matter how many times you
# step on it.
#
# The always-valid p-value is the running minimum of 1 / Lambda_n, where
#   Lambda_n = sqrt(V/(V+tau^2)) * exp(tau^2 * diff^2 / (2 V (V+tau^2)))
# V is the variance of the current difference estimate and tau^2 is the
# width of the mixing prior over plausible effect sizes.
#
# WHY THIS MATTERS: Experimentation platforms whose dashboards update
# continuously invite daily peeking; without sequential methods, a large
# share of "significant" results are noise that disappears after rollout.



## TASK 1 — Load Data and Compute Baseline


In [ ]:
print_banner("MLFP02 Exercise 7.3: Sequential Testing (mSPRT)")

# TODO: Load the experiment, split control vs ONE arm, check its SRM.
# Hint: load_experiment(); split_groups(experiment, ANALYSIS_ARM);
#   compute_srm(n_control, n_treatment) (defaults = designed shares)
experiment = ____
control, treatment = ____
srm_p = ____
if srm_p < 0.01:
    raise RuntimeError(f"SRM on control vs {ANALYSIS_ARM} (p={srm_p:.2g}) — stop")
# TODO: Extract revenue arrays and compute the naive A/B baseline.
# Hint: get_revenue_arrays(control, treatment); naive_ab(y_c, y_t)
y_c, y_t = ____
baseline = ____
se_naive = baseline["se"]

print(f"  Data loaded: {experiment.shape[0]:,} rows; analysing control vs {ANALYSIS_ARM}")
print(f"  Pairwise SRM p={srm_p:.3f} (OK)")
print(f"  Baseline SE: ${se_naive:.2f}")
print(f"  Baseline lift: ${baseline['lift']:.2f}")



### Checkpoint 1


In [ ]:
assert se_naive > 0, "Baseline SE must be positive"
print("\n>>> Checkpoint 1 passed -- data loaded and baseline computed\n")



## TASK 2 — Sequential Testing: mSPRT Day by Day


In [ ]:
print(f"\n=== Sequential Testing (mSPRT) ===")

# TODO: Set the mSPRT hyperparameter and compute day-by-day p-values.
# Hint: tau_sq is typically se_naive**2;
#   msprt_sequential_pvalues(experiment, tau_sq=..., treatment_arm=ANALYSIS_ARM)
tau_sq = ____
sequential_results = ____

print(f"{'Day':>4} {'n':>8} {'Lift':>10} {'p (fixed)':>12} {'p (mSPRT)':>12}")
print("-" * 52)
step = max(1, len(sequential_results) // 10)
for r in sequential_results[::step]:
    print(
        f"{r['day']:>4} {int(r['n']):>8,} ${r['lift']:>8.2f} "
        f"{r['p_fixed']:>12.6f} {r['p_sequential']:>12.6f}"
    )

early_sig_fixed = sum(1 for r in sequential_results if r["p_fixed"] < 0.05)
early_sig_seq = sum(1 for r in sequential_results if r["p_sequential"] < 0.05)
print(f"\nDays with p < 0.05 (fixed):      {early_sig_fixed}/{len(sequential_results)}")
print(f"Days with p < 0.05 (sequential): {early_sig_seq}/{len(sequential_results)}")
# INTERPRETATION: Fixed p-values may cross 0.05 early (giving false
# confidence), while mSPRT p-values stay conservative until real
# evidence accumulates. If a product manager stopped at the first
# fixed-p < 0.05, they might ship a non-effect.

# Compute the mixture likelihood ratio yourself at the final look
# (all data): diff = full-sample lift, V = se_naive^2.
v_final = se_naive**2
diff_final = baseline["lift"]
# TODO: Implement Lambda_n from the THEORY block with np.sqrt and np.exp.
lambda_final = ____
print(f"\nFinal look: Lambda = {lambda_final:.3g}, 1/Lambda = {1 / lambda_final:.3g}")



### Checkpoint 2


In [ ]:
assert len(sequential_results) > 0, "Must have sequential results"
assert np.isclose(lambda_final, msprt_lambda(diff_final, v_final, tau_sq), rtol=1e-9), (
    "Your Lambda should match the reference msprt_lambda helper"
)
seq_ps = [r["p_sequential"] for r in sequential_results]
assert all(b <= a for a, b in zip(seq_ps, seq_ps[1:])), "Always-valid p must never increase"
for r in sequential_results:
    assert 0 <= r["p_sequential"] <= 1, "Sequential p-values must be valid"
print("\n>>> Checkpoint 2 passed -- sequential testing completed\n")



## TASK 3 — Peeking Problem Simulation


In [ ]:
# Simulate experiments with NO real effect to show how peeking inflates
# the false positive rate.

print(f"\n=== Peeking Problem Simulation ===")

# TODO: Run the peeking simulation using the shared helper.
# Hint: simulate_peeking(n_sims=1000, n_per_sim=2000, n_checks=20, seed=42)
#   returns a dict with rate_no_peek and rate_fixed_peek.
peek_results = ____

print(f"Simulations: {int(peek_results['n_sims']):,} (all with NO real effect)")
print(f"Peeks per experiment: {int(peek_results['n_checks'])}")
print(f"\nFalse positive rates:")
print(f"  No peeking (test at end):   {peek_results['rate_no_peek']:.1%} (target: 5%)")
print(
    f"  Peeking with fixed p:       {peek_results['rate_fixed_peek']:.1%} (inflated!)"
)
print(
    f"  Inflation factor:           "
    f"{peek_results['rate_fixed_peek'] / peek_results['rate_no_peek']:.1f}x"
)
# INTERPRETATION: Read the simulated rate above — with 20 looks it lands
# around 25%, not 5%. It is well below the 64% an "independent tests"
# calculation would predict, because each look re-uses earlier data.
# Sequential testing (mSPRT) is the correct way to monitor experiments.



### Checkpoint 3


In [ ]:
assert (
    peek_results["rate_fixed_peek"] > peek_results["rate_no_peek"]
), "Peeking must inflate false positive rate"
print("\n>>> Checkpoint 3 passed -- peeking problem demonstrated\n")



## TASK 4 — Visualise: Fixed vs Sequential p-Values Over Time


In [ ]:
fig = go.Figure()
days_seq = [r["day"] for r in sequential_results]
fig.add_trace(
    go.Scatter(
        x=days_seq,
        y=[r["p_fixed"] for r in sequential_results],
        name="Fixed p-value",
    )
)
fig.add_trace(
    go.Scatter(
        x=days_seq,
        y=[r["p_sequential"] for r in sequential_results],
        name="mSPRT p-value",
    )
)
fig.add_hline(y=0.05, line_dash="dash", annotation_text="alpha=0.05")
fig.update_layout(
    title="Sequential Testing: Fixed vs mSPRT p-values",
    xaxis_title="Day",
    yaxis_title="p-value",
    yaxis_type="log",
)
out_path = OUTPUT_DIR / "sequential_pvalues.html"
fig.write_html(str(out_path))
print(f"\nSaved: {out_path}")

# Peeking problem visualisation
fig2 = go.Figure()
categories = ["No peeking", f"Peeking ({int(peek_results['n_checks'])} looks, fixed p)"]
rates = [
    peek_results["rate_no_peek"],
    peek_results["rate_fixed_peek"],
]
colours = ["green", "red"]
fig2.add_trace(
    go.Bar(
        x=categories,
        y=rates,
        marker_color=colours,
        text=[f"{r:.1%}" for r in rates],
        textposition="auto",
    )
)
fig2.add_hline(y=0.05, line_dash="dash", annotation_text="Nominal alpha=5%")
fig2.update_layout(
    title="Peeking Problem: False Positive Rate Inflation",
    yaxis_title="False Positive Rate",
    yaxis_tickformat=".0%",
)
out_path2 = OUTPUT_DIR / "peeking_problem.html"
fig2.write_html(str(out_path2))
print(f"Saved: {out_path2}")



## APPLY — A Singapore Ride-Hailing Platform: Continuous Monitoring


In [ ]:
# Scenario (illustrative figures): a ride-hailing platform concludes ~50
# experiments a month. Dashboards update continuously and product
# managers check results about 20 times per experiment.
#
# Worst case — none of the 50 changes truly works:
#   - With peeking, each experiment has the SIMULATED false-positive rate
#     from Task 3 of being declared a winner.
#   - With mSPRT, the false-positive rate is at most 5% (alpha).
#   - Each false positive ships a non-effect: development, rollback and
#     re-test cost an assumed S$100K.
#   - Trade-off: mSPRT needs more data than a single fixed-horizon test.

print(f"\n--- Singapore Application: Ride-Hailing Experiment Monitoring ---")
n_experiments_per_month = 50  # illustrative
# TODO: Expected false positives per month under peeking (simulated rate).
fp_peeking = ____
fp_msprt_bound = n_experiments_per_month * 0.05  # upper bound under H0
cost_per_fp = 100_000  # S$, illustrative
print(f"Experiments concluded per month: {n_experiments_per_month}")
print(f"False positives per month (peeking, simulated rate): ~{fp_peeking:.1f}")
print(f"False positives per month (mSPRT, at most):          ~{fp_msprt_bound:.1f}")
print(f"Cost per false positive (assumed): S${cost_per_fp:,}")
print(f"Annual waste (peeking): S${fp_peeking * cost_per_fp * 12:,.0f}")
print(f"Annual waste (mSPRT, at most): S${fp_msprt_bound * cost_per_fp * 12:,.0f}")
print(f"Annual savings (at least): S${(fp_peeking - fp_msprt_bound) * cost_per_fp * 12:,.0f}")



## LOG — ExperimentTracker


In [ ]:
async def log_sequential_results():
    db = "sqlite:///mlfp02_experiments.db"
    tracker = await ExperimentTracker.create(store_url=db)

    exp_id = "mlfp02_ex7_sequential_testing"

    async with tracker.track(experiment=exp_id, run_name="msprt_analysis") as run:
        await run.log_params(
            {
                "sequential_method": "mSPRT",
                "treatment_arm": ANALYSIS_ARM,
                "tau_sq": str(float(tau_sq)),
                "n_peek_sims": "1000",
                "n_checks": "20",
            }
        )
        await run.log_metrics(
            {
                "days_sig_fixed": float(early_sig_fixed),
                "days_sig_sequential": float(early_sig_seq),
                "fp_rate_no_peek": float(peek_results["rate_no_peek"]),
                "fp_rate_peeking": float(peek_results["rate_fixed_peek"]),
            }
        )
    print(f"\nLogged sequential testing run")
    await tracker.close()


await log_sequential_results()



### Checkpoint 4


In [ ]:
print("\n>>> Checkpoint 4 passed -- visualisation and logging complete\n")



## REFLECTION


- mSPRT: always-valid p-values for safe experiment monitoring
  - Peeking problem: {int(peek_results['n_checks'])} correlated peeks inflate alpha from 5% to ~{peek_results['rate_fixed_peek']:.0%} (simulated)
  - The mixture likelihood ratio Lambda and the running-minimum p-value
  - Fixed vs sequential p-value trajectories
  - tau_sq hyperparameter: set to baseline SE^2
  - Why dashboards with live p-values need sequential methods

  NEXT: In 04_diff_in_diff.py, you'll learn to estimate causal effects
  from observational data when randomisation is not possible.


In [ ]:
print("=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)

print("\n>>> Exercise 7.3 complete -- Sequential Testing with mSPRT")

